In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

print("=" * 80)
print("AIRPLANE CRASHES AND FATALITIES ANALYSIS (up to 2023)")
print("=" * 80)
print("\n1. DATA IMPORT AND CLEANING")
print("-" * 80)

# Load the local dataset
local_csv = "Airplane_Crashes_and_Fatalities_Since_1908_t0_2023.csv"

if os.path.exists(local_csv):
    print(f"Loading local dataset: {local_csv}")
    df = pd.read_csv(local_csv)
    print(f"✓ Dataset loaded successfully")
else:
    raise FileNotFoundError(f"Dataset not found: {local_csv}")

print(f"\nDataset shape: {df.shape}")
print(f"\nColumn names and types:")
print(df.dtypes)
print(f"\nFirst few rows:")
print(df.head())

# Data cleaning
print("\n" + "-" * 80)
print("Data Cleaning:")
print("-" * 80)

# Display missing values
print(f"\nMissing values:")
missing = df.isnull().sum()
print(missing[missing > 0])

# Convert date columns if they exist
date_columns = [col for col in df.columns if 'date' in col.lower()]
for col in date_columns:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')
        print(f"✓ Converted '{col}' to datetime")

# Handle missing values
for col in df.columns:
    if df[col].dtype in ['int64', 'float64']:
        if df[col].isnull().sum() > 0:
            df[col].fillna(df[col].median(), inplace=True)
            print(f"✓ Filled missing numeric values in '{col}' with median")
    elif df[col].dtype == 'object':
        if df[col].isnull().sum() > 0:
            df[col].fillna('Unknown', inplace=True)
            print(f"✓ Filled missing text values in '{col}' with 'Unknown'")

print(f"\n✓ Data cleaning complete. Final shape: {df.shape}")


In [ ]:
print("\n" + "=" * 80)
print("2. EXPLORATORY DATA ANALYSIS")
print("=" * 80)

# Identify numeric and categorical columns
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()

print(f"\nNumeric columns: {numeric_cols}")
print(f"Categorical columns: {categorical_cols}")

# Basic statistics
print("\n" + "-" * 80)
print("Basic Statistics:")
print("-" * 80)
print(f"\nTotal number of records: {len(df)}")
print(f"\nDescriptive Statistics for numeric columns:")
print(df[numeric_cols].describe())

# Analyze crashes over time
print("\n" + "-" * 80)
print("Crashes Over Time:")
print("-" * 80)

date_col = None
for col in date_columns:
    if col in df.columns and df[col].dtype == 'datetime64[ns]':
        date_col = col
        break

if date_col:
    # Extract year
    df['Year'] = df[date_col].dt.year
    crashes_per_year = df['Year'].value_counts().sort_index()
    print(f"\nCrashes per year (first 10 years with data):")
    print(crashes_per_year.head(10))
    
    # Identify decades
    df['Decade'] = (df['Year'] // 10 * 10).astype(str) + 's'
    crashes_per_decade = df['Decade'].value_counts().sort_index()
    print(f"\nCrashes per decade:")
    print(crashes_per_decade)
    
    # Fatalities over time
    for col in numeric_cols:
        if 'fatal' in col.lower() or 'death' in col.lower():
            print(f"\n{col} statistics by decade:")
            fatalities_by_decade = df.groupby('Decade')[col].agg(['sum', 'mean', 'median', 'max'])
            print(fatalities_by_decade)
else:
    print("No datetime column found for time analysis")

# Categorical analysis
print("\n" + "-" * 80)
print("Categorical Variables Analysis:")
print("-" * 80)

for col in categorical_cols[:5]:  # Analyze first 5 categorical columns
    if df[col].nunique() <= 20:  # Only if reasonable number of categories
        print(f"\n{col} - Top categories:")
        print(df[col].value_counts().head())


In [ ]:
print("\n" + "=" * 80)
print("3. STATISTICAL ANALYSIS")
print("=" * 80)

# Find fatality column
fatality_col = None
for col in numeric_cols:
    if 'fatal' in col.lower() or 'death' in col.lower():
        fatality_col = col
        break

if not fatality_col:
    # Check for survival rate column
    for col in numeric_cols:
        if 'fatal' in col.lower():
            fatality_col = col
            break

print(f"\n" + "-" * 80)
print("Distribution of Fatalities:")
print("-" * 80)

if fatality_col and fatality_col in df.columns:
    # Remove NaN values for analysis
    fatalities = df[fatality_col].dropna()
    
    print(f"\n{fatality_col} Statistics:")
    print(f"Mean: {fatalities.mean():.2f}")
    print(f"Median: {fatalities.median():.2f}")
    print(f"Std Dev: {fatalities.std():.2f}")
    print(f"Min: {fatalities.min():.0f}")
    print(f"Max: {fatalities.max():.0f}")
    print(f"Skewness: {stats.skew(fatalities):.4f}")
    print(f"Kurtosis: {stats.kurtosis(fatalities):.4f}")
    
    # Normality test
    statistic, p_value = stats.normaltest(fatalities)
    print(f"\nNormality Test (D'Agostino-Pearson):")
    print(f"Test Statistic: {statistic:.4f}")
    print(f"P-Value: {p_value:.6f}")
    if p_value < 0.05:
        print("Conclusion: Fatality distribution is NOT normally distributed (reject null hypothesis)")
    else:
        print("Conclusion: Fatality distribution appears to be normally distributed")

# Hypothesis testing - comparing different time periods
if date_col and 'Decade' in df.columns:
    print(f"\n" + "-" * 80)
    print("Hypothesis Testing - Fatalities by Decade:")
    print("-" * 80)
    
    # Get decades with enough data
    decades = df['Decade'].unique()
    decade_groups = []
    
    for decade in sorted(decades):
        if decade != 'nan':
            decade_data = df[df['Decade'] == decade][fatality_col].dropna()
            if len(decade_data) > 5:  # Only if we have enough samples
                decade_groups.append(decade_data)
                print(f"\n{decade}: n={len(decade_data)}, mean={decade_data.mean():.2f}, median={decade_data.median():.2f}")
    
    # ANOVA test
    if len(decade_groups) >= 2:
        f_stat, p_value = stats.f_oneway(*decade_groups)
        print(f"\nANOVA Test (comparing fatalities across decades):")
        print(f"F-Statistic: {f_stat:.4f}")
        print(f"P-Value: {p_value:.6f}")
        if p_value < 0.05:
            print("Conclusion: There are significant differences in fatalities across decades")
        else:
            print("Conclusion: No significant differences in fatalities across decades")


In [ ]:
print("\n" + "=" * 80)
print("4. VISUALIZATION - PART 1: Time Series and Distributions")
print("=" * 80)

# Create a figure with multiple subplots
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Plot 1: Crashes per year
if date_col and 'Year' in df.columns:
    crashes_per_year = df['Year'].value_counts().sort_index()
    axes[0, 0].plot(crashes_per_year.index, crashes_per_year.values, linewidth=2, color='darkblue', marker='o')
    axes[0, 0].fill_between(crashes_per_year.index, crashes_per_year.values, alpha=0.3)
    axes[0, 0].set_xlabel('Year')
    axes[0, 0].set_ylabel('Number of Crashes')
    axes[0, 0].set_title('Airplane Crashes Over Time')
    axes[0, 0].grid(True, alpha=0.3)

# Plot 2: Crashes per decade
if 'Decade' in df.columns:
    crashes_per_decade = df['Decade'].value_counts().sort_index()
    axes[0, 1].bar(crashes_per_decade.index, crashes_per_decade.values, color='steelblue', edgecolor='black')
    axes[0, 1].set_xlabel('Decade')
    axes[0, 1].set_ylabel('Number of Crashes')
    axes[0, 1].set_title('Crashes by Decade')
    axes[0, 1].tick_params(axis='x', rotation=45)
    axes[0, 1].grid(True, alpha=0.3, axis='y')

# Plot 3: Distribution of fatalities (histogram)
if fatality_col and fatality_col in df.columns:
    fatalities = df[fatality_col].dropna()
    axes[1, 0].hist(fatalities, bins=50, color='salmon', edgecolor='black', alpha=0.7)
    axes[1, 0].set_xlabel('Number of Fatalities')
    axes[1, 0].set_ylabel('Frequency')
    axes[1, 0].set_title(f'Distribution of {fatality_col}')
    axes[1, 0].axvline(fatalities.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {fatalities.mean():.0f}')
    axes[1, 0].axvline(fatalities.median(), color='green', linestyle='--', linewidth=2, label=f'Median: {fatalities.median():.0f}')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3, axis='y')

# Plot 4: Box plot of fatalities by decade
if 'Decade' in df.columns and fatality_col in df.columns:
    decade_data = []
    decades_list = []
    for decade in sorted(df['Decade'].unique()):
        if decade != 'nan':
            data = df[df['Decade'] == decade][fatality_col].dropna()
            if len(data) > 0:
                decade_data.append(data)
                decades_list.append(decade)
    
    if decade_data:
        bp = axes[1, 1].boxplot(decade_data, labels=decades_list, patch_artist=True)
        for patch in bp['boxes']:
            patch.set_facecolor('lightblue')
        axes[1, 1].set_xlabel('Decade')
        axes[1, 1].set_ylabel('Number of Fatalities')
        axes[1, 1].set_title('Fatalities Distribution by Decade')
        axes[1, 1].tick_params(axis='x', rotation=45)
        axes[1, 1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("✓ Time series and distribution visualizations complete")
